# Benchmark-relative mean-variance portfolio with a PCA risk model

For each rebalance date $t$ (the signal dates), this notebook builds target weights for two variants that differ only in the benchmark:

| Variant | Benchmark $b$ |
|---|---|
| `cap` | market-cap weights of the investable (compliant) universe |
| `ew`  | equal weights of the same investable universe |
| `spx` | S&P 500 proxy: cap weights of the top 500 names by market cap, **compliant or not**; non-compliant names can't be held ($w_i = 0$) |

**Pipeline**

1. Load the signal $\hat y$ (`pred_target_6m` by default: a prediction of the cross-sectionally z-scored log return over the next $h = 6$ months; set `SIGNAL_COL` / `HORIZON_M` for another horizon)
2. Compliance universe (point-in-time)
3. Daily prices → daily and monthly log returns
4. Scale $s$ → expected returns $\mu = s\,\hat y$
5. Market caps (point-in-time) → benchmarks
6. PCA risk model: $\Sigma = B F B^\top + D$
7. Optimizer: projected (accelerated) gradient with active-weight box, budget, 100-name cap, minimum weight
8. Run over all dates, diagnostics, export weights

**Parameters agreed**

| Parameter | Value |
|---|---|
| Signal horizon $h$ | 6 months (`pred_target_6m`); portfolio still rebalanced monthly |
| Risk / $s$ window | 2 years (504 trading days / 24 signal months) |
| Factors $k$ | number of eigenvalues above the Marchenko–Pastur edge, each date |
| Minimum history | 252 daily returns in the window |
| Specific-variance floor $\psi_{\min}$ | 0.2 (check the plot in §6) |
| Calibration $\beta_t$ | slope of realized vs predicted z, past data only (§4b) |
| Risk aversion $\lambda$ | 3 (locked after comparing 5 and 3) |
| Turnover cost $\kappa$ | 10 bps one-way, spread over the signal horizon: $\kappa = 10\text{ bps}/h$ (locked after comparing 0, 10 bps, 10 bps/6) |
| Active cap | $\lvert w_i - b_i\rvert \le 4\%$ (locked after comparing 2, 3, 4%) |
| Max names | 100 |
| Minimum position | 0.5% |
| Long-only, fully invested | $w_i \ge 0,\ \sum_i w_i = 1$ |

## 0. Configuration

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
OUT_DIR = Path("../output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SIGNAL_COL = "pred_target_6m"   # model output, z-score units
HORIZON_M = 6                   # horizon of SIGNAL_COL in months (1 for pred_target_1m, 3 for 3m, 6 for 6m)

H = 21                 # trading days per month (daily -> monthly scaling)
WINDOW_DAYS = 504      # 2 years of daily returns for the risk model
S_WINDOW_MONTHS = 24   # 2 years of months for s
MIN_HISTORY_DAYS = 252 # min non-missing daily returns in the window
K_MIN = 1              # k is set each date = number of eigenvalues above the MP edge (at least K_MIN)
PSI_FLOOR = 0.2        # floor on the specific share of variance

Z_CLIP = 5.0           # clip realized z when estimating the calibration slope beta
MIN_CAL_MONTHS = 6     # minimum past months for beta

LAMBDA = 3.0           # risk aversion on active risk
ACTIVE_CAP = 0.04      # |w_i - b_i| <= 4%
TRADE_COST = 0.001                        # one-way cost per unit of weight traded (10 bps)
TURNOVER_COST = TRADE_COST / HORIZON_M    # kappa: cost spread over the signal horizon (mu is monthly); 0 = no penalty
SPX_N = 500            # size of the S&P 500 proxy benchmark
MAX_NAMES = 100
MIN_WEIGHT = 0.005

START_DATE = "2012-07-31"   # first date with a 6-month s and beta; None = first signal date
END_DATE = "2025-08-31"     # None = last signal date
DIAG_DATE = None       # date for the detailed PCA plots; None = last rebalance date

# chart style (one axis, recessive grid, thin lines)
C_CAP, C_EW, C_SPX, C_REF = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
plt.rcParams.update({
    "figure.figsize": (9, 4), "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlesize": 12,
    "axes.titlecolor": "#0b0b0b", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "xtick.color": "#52514e", "ytick.color": "#52514e", "lines.linewidth": 2,
    "legend.frameon": False,
})

## 1. Signal

$\hat y_{i,t}$ is the Ridge prediction of the cross-sectionally z-scored log return over the next $h$ months:

$$y_{i,t} = \frac{R_{i,t\to t+h} - \bar R_{t\to t+h}}{\operatorname{std}_j\!\left(R_{j,t\to t+h}\right)}, \qquad R_{i,t\to t+h} = \ln\frac{P_{i,t+h}}{P_{i,t}}$$

It is already shrunk toward 0 by the model, so it is used as is (no IC scaling).
If a `(symbol, date)` appears in several `run_id`s, the latest `run_id` is kept.

In [ ]:
sig = pd.read_parquet(DATA_DIR / "model_signal.parquet",
                      columns=["symbol", "date", "period_id", "run_id", SIGNAL_COL])
sig["symbol"] = sig["symbol"].astype(str)

# instrument-type filter (same as the backtest): drop preferreds/units/warrants and 5-letter F/Y listings
bad = sig["symbol"].str.contains(r"[.\-]") | ((sig["symbol"].str.len() == 5) & sig["symbol"].str[-1].isin(["F", "Y"]))
print(f"instrument filter: dropped {sig.loc[bad, 'symbol'].nunique()} symbols")
sig = sig[~bad]

# rebalance date = calendar month end of period_id (same as the backtest's period_date)
sig["avail_date"] = pd.to_datetime(sig["date"]).dt.normalize()
sig["date"] = pd.to_datetime(sig["period_id"].str.replace("_period_", ""), format="%Y-%m") + pd.offsets.MonthEnd(0)
n_after = (sig["avail_date"] > sig["date"]).sum()
print(f"predictions available after period end (look-ahead, dropped): {n_after}")
sig = sig[sig["avail_date"] <= sig["date"]]

n_dup = sig.duplicated(["symbol", "date"]).sum()
sig = (sig.sort_values(["run_id", "avail_date"])
          .drop_duplicates(["symbol", "date"], keep="last")
          .dropna(subset=[SIGNAL_COL]))
print(f"duplicate (symbol, period) rows dropped: {n_dup}")

rebal_dates = sorted(sig["date"].unique())
if START_DATE: rebal_dates = [d for d in rebal_dates if d >= pd.Timestamp(START_DATE)]
if END_DATE:   rebal_dates = [d for d in rebal_dates if d <= pd.Timestamp(END_DATE)]
rebal_dates = [pd.Timestamp(d) for d in rebal_dates]

sig_by_date = {d: g.set_index("symbol")[SIGNAL_COL] for d, g in sig.groupby("date")}
sig_symbols = sorted(sig["symbol"].unique())
per_date = sig.groupby("date").size()
print(f"rebalance dates: {len(rebal_dates)}  ({rebal_dates[0].date()} -> {rebal_dates[-1].date()})")
print(f"names per date: median {per_date.median():.0f}, min {per_date.min()}, max {per_date.max()}")
print(f"cross-sectional std of y_hat (median over dates): {sig.groupby('date')[SIGNAL_COL].std().median():.3f}")

## 2. Compliance universe

The dict is keyed by month (`YYYY-MM`) with an `as_of` date. At rebalance date $t$ we use the latest entry with `as_of` $\le t$. Only `compliant` is investable (`doubtful` is excluded). Non-compliant stocks are still used to estimate the PCA factors. Same rule as the backtest's `apply_shariah_filtering_by_date`.

In [ ]:
with open(DATA_DIR / "compliance_dict.json") as f:
    comp_raw = json.load(f)
comp = sorted((pd.Timestamp(v["as_of"]), set(map(str, v["compliant"]))) for v in comp_raw.values())
comp_dates = [d for d, _ in comp]

def compliant_at(t):
    i = np.searchsorted(comp_dates, t, side="right") - 1
    return comp[i][1] if i >= 0 else set()

cov = pd.DataFrame({
    "compliant": [len(compliant_at(t)) for t in rebal_dates],
    "compliant_with_signal": [len(compliant_at(t) & set(sig_by_date[t].index)) for t in rebal_dates],
}, index=rebal_dates)

fig, ax = plt.subplots()
ax.plot(cov.index, cov["compliant"], color=C_CAP, label="Compliant")
ax.plot(cov.index, cov["compliant_with_signal"], color=C_EW, label="Compliant with a signal")
ax.set_title("Compliance universe size"); ax.set_ylabel("Number of stocks"); ax.legend()
plt.show()
cov.tail()

## 3. Prices and returns

Daily log returns $r_{i,d} = \ln(P_{i,d}/P_{i,d-1})$ on the trading-day calendar of the file. A missing price makes that day's return missing (no return is stitched across gaps).

**Price-outlier rule (same as the backtest):** on forward-filled prices, the first day with a simple return beyond ±75% marks the ticker; from that date on it is excluded from both the risk model and the investable set.

Monthly log returns use the last trading day of each month.

**`close` must be split/dividend adjusted.** The check below lists the largest daily moves: a ±50–90% one-day move on a large cap is almost always an unadjusted split.

In [ ]:
t0 = time.time()
first_needed = rebal_dates[0] - pd.Timedelta(days=800)
tbl = pq.read_table(DATA_DIR / "historical_prices.parquet",
                    columns=["symbol", "date", "close"],
                    filters=[("symbol", "in", sig_symbols), ("date", ">=", first_needed)])
px = tbl.to_pandas()
del tbl
px["date"] = pd.to_datetime(px["date"]).dt.normalize()
px["symbol"] = px["symbol"].astype(str)
px = px.drop_duplicates(["symbol", "date"], keep="last")
close = px.pivot(index="date", columns="symbol", values="close").sort_index()
del px
close = close.where(close > 0)
logret = np.log(close).diff().iloc[1:]
print(f"close matrix: {close.shape}  loaded in {time.time() - t0:.0f}s")

# first outlier date per ticker (backtest rule: ffilled prices, |simple return| > 75%)
_r = close.ffill().pct_change()
_m = (_r < -0.75) | (_r > 0.75)
first_outlier = _m.loc[:, _m.any()].idxmax()
del _r, _m
print(f"tickers with an outlier move: {len(first_outlier)}")

def clean_at(t, names):
    """names not excluded by the outlier rule at t (excluded if first outlier date <= t)."""
    fo = first_outlier.reindex(list(names))
    return set(fo.index[fo.isna() | (fo > t)])

ext = logret.stack()
ext = ext[ext.abs() > np.log(1.5)]
print(f"daily |log return| > ln(1.5): {len(ext)} of {logret.notna().sum().sum()} observations")
print(ext.abs().sort_values(ascending=False).head(15).rename("abs_logret"))

In [ ]:
# monthly log returns from the last trading day of each month
month_end_idx = close.index.to_series().groupby(close.index.to_period("M")).max()
close_m = close.loc[month_end_idx.values]
close_m.index = month_end_idx.index            # PeriodIndex (monthly)
logret_m = np.log(close_m).diff().iloc[1:]
logret_m.shape

## 4. From $\hat y$ to expected returns

The target was divided by one number per month (the cross-sectional std), so the prediction is converted back with one number:

$$s_m = 1.4826\cdot\operatorname{median}_i\left|\,R_{i,m\to m+h} - \operatorname{median}_j R_{j,m\to m+h}\right| \quad\text{over the stocks with a signal in month } m$$

$s_m$ is only known once month $m+h$ has ended. At rebalance date $t$ we average the last 24 known values (at least 6):

$$s_t = \operatorname{mean}\{\, s_m : m + h \le t \,\}_{\text{last }24}$$

$s_t\,\hat y_{i,t}$ is an $h$-month return. The portfolio is rebalanced monthly and $\Sigma$ is monthly, so the expected return per month is $s_t\,\hat y_{i,t}/h$ (the edge is assumed to accrue evenly over the horizon).

**Why a robust std:** a plain std squares deviations, so a few broken tickers (failed-bank preferreds and OTC lines with log moves of 5–11) can multiply it by 4. The MAD-based estimate equals the std for normal-looking data but ignores a handful of extreme names. Tickers excluded by the >75% outlier rule are also removed.

$\mu_{i,t}$ (§4b) is the expected monthly log return of stock $i$ **relative to the average stock**. The average itself is the same for all stocks and drops out because $\sum_i w_i = \sum_i b_i = 1$.

In [ ]:
sig_syms_by_period = {pd.Timestamp(d).to_period("M"): set(s.index) for d, s in sig_by_date.items()}

# forward h-month log return per signal month m: fwd.loc[m] = sum of monthly log returns over m+1 .. m+h
fwd = logret_m.rolling(HORIZON_M).sum().shift(-HORIZON_M)

s_h = {}
for m in fwd.index:
    names = sig_syms_by_period.get(m)
    if not names:
        continue
    names = clean_at((m + HORIZON_M).to_timestamp(how="end"), names)
    x = fwd.loc[m, fwd.columns.intersection(list(names))].dropna()
    if len(x) > 30:
        s_h[m] = 1.4826 * (x - x.median()).abs().median()
s_h = pd.Series(s_h).sort_index()          # indexed by signal month m; known once m + h has ended

def s_at(t):
    P = pd.Timestamp(t).to_period("M")
    w = s_h[s_h.index <= P - HORIZON_M].tail(S_WINDOW_MONTHS)
    return w.mean() if len(w) >= 6 else np.nan

s_series = pd.Series({t: s_at(t) for t in rebal_dates})
fig, ax = plt.subplots()
ax.plot(s_h.index.to_timestamp(how="end"), s_h.values, color=C_REF, linewidth=1, label="Single signal month $s_m$")
ax.plot(s_series.index, s_series.values, color=C_CAP, label="Average of last 24 known $s_m$ (used)")
ax.set_title(f"Cross-sectional std of {HORIZON_M}-month log returns"); ax.set_ylabel(f"Std ({HORIZON_M}-month)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}")); ax.legend()
plt.show()
s_series.describe()

## 4b. Calibration: how big are realized returns per unit of prediction?

If $\hat y$ were calibrated, a stock predicted at $\hat y = 0.3$ would on average realize $z = 0.3$. In practice predictions are too spread out, so we measure the slope between predicted and realized z on **past data only**.

For each signal month $m$ and stock $i$ with a signal in $m$ (outlier tickers excluded), with the realized return over the next $h$ months:

$$z_{i,m} = \operatorname{clip}\!\left(\frac{R_{i,m\to m+h} - \bar R_{m\to m+h}}{s_{m}},\ \pm 5\right), \qquad \tilde y_{i,m} = \hat y_{i,m} - \bar{\hat y}_m$$

At rebalance date $t$, pooling all signal months whose $h$-month return is complete by $t$, i.e. $m + h \le t$ (expanding window, at least 6 months):

$$\beta_t = \frac{\sum_{m + h \le t}\sum_i \tilde y_{i,m}\, z_{i,m}}{\sum_{m + h \le t}\sum_i \tilde y_{i,m}^2}, \qquad \beta_t \leftarrow \max(\beta_t, 0)$$

With $h > 1$ consecutive months overlap (their windows share $h-1$ months), which makes $\beta_t$ smoother but doesn't bias it.

The mean (or any per-month constant) in $z$ cancels because $\tilde y$ sums to zero each month. $\beta_t = 1$: calibrated. $\beta_t < 1$: overconfident.

**Expected returns used by the optimizer:**

$$\mu_{i,t} = \frac{\beta_t\, s_t\, \hat y_{i,t}}{h}$$

Scaling $\mu$ by $\beta$ is the same as dividing $\lambda$ by $\beta$; keeping them separate means $\lambda$ stays your risk preference and $\beta$ is measured from data.

In [ ]:
num, den, n_pairs = {}, {}, {}
for d, yh in sig_by_date.items():
    m = pd.Timestamp(d).to_period("M")
    if m not in fwd.index or m not in s_h.index:
        continue
    names = clean_at((m + HORIZON_M).to_timestamp(how="end"), yh.index)
    r = fwd.loc[m].reindex(sorted(names)).dropna()
    if len(r) < 30:
        continue
    z = ((r - r.mean()) / s_h[m]).clip(-Z_CLIP, Z_CLIP)
    y = yh.reindex(z.index)
    y = y - y.mean()
    num[m], den[m], n_pairs[m] = float((y * z).sum()), float((y ** 2).sum()), len(z)
cal = pd.DataFrame({"num": num, "den": den, "n": n_pairs}).sort_index()
cal["beta_month"] = cal["num"] / cal["den"]

def beta_at(t):
    m = pd.Timestamp(t).to_period("M")
    past = cal[cal.index <= m - HORIZON_M]  # h-month return complete by t
    if len(past) < MIN_CAL_MONTHS:
        return np.nan
    return max(past["num"].sum() / past["den"].sum(), 0.0)

beta_series = pd.Series({t: beta_at(t) for t in rebal_dates})

fig, ax = plt.subplots()
ax.plot(cal.index.to_timestamp(how="end"), cal["beta_month"], color=C_REF, linewidth=1, label="Single month slope")
ax.plot(beta_series.index, beta_series.values, color=C_CAP, label=r"Expanding $\beta_t$ (used)")
ax.axhline(0, color=C_REF, linewidth=0.8)
ax.set_title("Calibration slope: realized z per unit of predicted z"); ax.legend()
plt.show()
print(f"months used: {len(cal)}, pairs: {int(cal['n'].sum())}")
beta_series.describe()

## 5. Market caps and benchmarks

**Eligibility (same as the backtest's `apply_marketcap_eligibility`):** 20-day rolling mean of market cap (file in \$MM) with hysteresis: a stock becomes eligible above \$5,000MM and stays eligible until it drops below \$4,000MM. Equivalently, eligibility is the most recent crossing signal: $+1$ when the rolling cap $> 5000$, $-1$ when $< 4000$.

**Currency:** the market-cap file stores foreign companies (ADRs) in their home currency (Telkom Indonesia in rupiah, Toyota in yen...), which makes them look thousands of times too big. Only names whose `marketCapCurrency` is USD in `company_profiles.parquet` keep a market cap; the others can't pass eligibility and get no benchmark weight (they still help estimate the PCA factors).

For each rebalance date $t$ we take, per stock, the latest `at_date` $\le t$ (within 10 days). Rows whose `snapshot_date` is after $t$ would be look-ahead; their count is printed.

Investable set $\mathcal I_t$ = has a signal $\cap$ compliant $\cap$ market-cap eligible $\cap$ not excluded by the outlier rule $\cap$ passes the risk-model history filter. Both variants use the same $\mathcal I_t$:

$$b^{\text{cap}}_i = \frac{\text{mcap}_i}{\sum_{j\in\mathcal I_t}\text{mcap}_j}, \qquad b^{\text{ew}}_i = \frac{1}{|\mathcal I_t|}$$

**S&P 500 proxy.** $\mathcal S_t$ = the 500 largest names by market cap among all stocks with a signal and a risk-model row at $t$, compliant or not:

$$b^{\text{spx}}_i = \frac{\text{mcap}_i\,\mathbb 1[i\in\mathcal S_t]}{\sum_{j\in\mathcal S_t}\text{mcap}_j}$$

Non-compliant names in $\mathcal S_t$ get $w_i = 0$, so their active weight is $-b_i$. The risk model penalizes that gap, which pushes the optimizer to fill it with compliant names that co-move with the excluded ones. Names dropped by the instrument filter (e.g. share classes with a dot, such as BRK.B) are not in the proxy.

In [ ]:
t0 = time.time()
# full history (no date filter): the hysteresis state depends on the whole path
tbl = pq.read_table(DATA_DIR / "market_cap.parquet",
                    columns=["symbol", "at_date", "market_capitalization", "snapshot_date"],
                    filters=[("symbol", "in", sig_symbols)])
mc = tbl.to_pandas(); del tbl
mc["symbol"] = mc["symbol"].astype(str)
mc["at_date"] = pd.to_datetime(mc["at_date"]).dt.normalize()
mc["snapshot_date"] = pd.to_datetime(mc["snapshot_date"]).dt.normalize()

mc = mc.sort_values(["symbol", "at_date"], kind="stable").reset_index(drop=True)
mc["rolling_mcap"] = (mc.groupby("symbol")["market_capitalization"]
                        .transform(lambda x: x.rolling(20, min_periods=1).mean()))
cross = pd.Series(np.select([mc["rolling_mcap"] > 5_000, mc["rolling_mcap"] < 4_000], [1.0, -1.0], np.nan), index=mc.index)
mc["eligible"] = cross.groupby(mc["symbol"]).ffill().eq(1.0)
del cross
mc = mc.dropna(subset=["market_capitalization"])
mc = mc[mc["market_capitalization"] > 0].sort_values("at_date")

# keep only market caps reported in USD (foreign ADRs are stored in local currency: IDR, JPY, KRW, HKD...)
prof = pd.read_parquet(DATA_DIR / "company_profiles.parquet", columns=["symbol", "marketCapCurrency", "snapshot_date"])
cur = prof.sort_values("snapshot_date").drop_duplicates("symbol", keep="last").set_index("symbol")["marketCapCurrency"]
usd_syms = set(cur.index[cur == "USD"])
n_sig = len(set(sig_symbols))
print(f"signal symbols: {n_sig} | USD market cap: {len(usd_syms & set(sig_symbols))} | "
      f"non-USD: {len(set(cur.index[cur != 'USD']) & set(sig_symbols))} | no profile: {len(set(sig_symbols) - set(cur.index))}")
mc = mc[mc["symbol"].isin(usd_syms)]

req = pd.DataFrame([(t, s) for t in rebal_dates for s in sig_by_date[t].index],
                   columns=["t", "symbol"]).sort_values("t")
req["t"] = req["t"].astype("datetime64[ns]")
mc["at_date"] = mc["at_date"].astype("datetime64[ns]")
mc_t = pd.merge_asof(req, mc, left_on="t", right_on="at_date", by="symbol",
                     direction="backward", tolerance=pd.Timedelta(days=10))
n_lookahead = (mc_t["snapshot_date"] > mc_t["t"]).sum()
print(f"look-ahead rows (snapshot_date > t): {n_lookahead}")
mc_t = mc_t[mc_t["snapshot_date"] <= mc_t["t"]].dropna(subset=["market_capitalization"])
mcap_all_by_date = {t: g.set_index("symbol")["market_capitalization"] for t, g in mc_t.groupby("t")}  # for the S&P proxy
mc_t = mc_t[mc_t["eligible"]]
mcap_by_date = {t: g.set_index("symbol")["market_capitalization"] for t, g in mc_t.groupby("t")}
print(f"eligible (date, symbol) pairs: {len(mc_t)}")
del mc
print(f"market caps matched: {len(mc_t)} of {len(req)} (date, symbol) pairs in {time.time() - t0:.0f}s")

## 6. PCA risk model

Window: the last $T = 504$ daily returns up to $t$. Universe: all stocks with a signal at $t$ (compliant or not), not excluded by the outlier rule, with at least 252 non-missing returns and a price on the last trading day $\le t$.

**Standardize.** With $\sigma_i$ the std of stock $i$'s daily returns over the window (available days only):

$$Z_{d,i} = \frac{r_{d,i} - \bar r_i}{\sigma_i}, \qquad \text{missing } Z_{d,i} \to 0$$

**PCA** of the correlation matrix $C = Z^\top Z/(T-1)$, done through the SVD $Z = U S_Z V^\top$ (no $N\times N$ matrix is built):

$$\lambda_m = \frac{s_{Z,m}^2}{T-1}, \qquad C \approx V_k \Lambda_k V_k^\top + \Psi, \qquad \psi_i = 1 - \sum_{m=1}^{k}\lambda_m v_{im}^2, \qquad \psi_i \leftarrow \max(\psi_i, \psi_{\min})$$

**Scale back and convert to monthly** ($H = 21$), with $S = \operatorname{diag}(\sigma)$:

$$\Sigma = H\, S\left(V_k\Lambda_k V_k^\top + \Psi\right) S = B F B^\top + D$$

$$B = S V_k \;(N\times k), \qquad F = H\Lambda_k \;(k\times k,\ \text{diagonal}), \qquad D = H\, \operatorname{diag}(\sigma_i^2 \psi_i)$$

**Number of factors (Marchenko–Pastur).** For pure noise with $q = N/T$, correlation eigenvalues stay below $\lambda_+ = (1+\sqrt q)^2$. Eigenvalues above $\lambda_+$ are distinguishable from noise, so at each date

$$k_t = \#\{m : \lambda_m > \lambda_+\}$$

No tuning: $k_t$ comes from the data and grows with the universe size.

In [ ]:
cal = logret.index

def risk_model(t, names):
    """PCA factor model at date t on the given names. Returns dict with B, F, D (monthly)."""
    idx = cal[cal <= t][-WINDOW_DAYS:]
    cols = logret.columns.intersection(sorted(clean_at(t, names)))
    R = logret.loc[idx, cols]
    has_px = close.loc[:t].iloc[-1].reindex(cols).notna()
    keep = (R.notna().sum() >= MIN_HISTORY_DAYS) & has_px
    R = R.loc[:, keep[keep].index]
    if R.shape[1] <= 30:
        return None
    sigma = R.std(ddof=1).to_numpy()
    Z = ((R - R.mean()) / R.std(ddof=1)).fillna(0.0).to_numpy()
    T = Z.shape[0]
    _, sv, Vt = np.linalg.svd(Z, full_matrices=False)
    eig = sv ** 2 / (T - 1)
    N = Z.shape[1]
    mp_edge = (1 + np.sqrt(N / T)) ** 2
    k = max(int((eig > mp_edge).sum()), K_MIN)
    V = Vt[:k].T
    lam = eig[:k]
    psi_raw = 1.0 - (V ** 2 * lam).sum(axis=1)
    psi = np.maximum(psi_raw, PSI_FLOOR)
    return dict(
        symbols=R.columns.to_numpy(), sigma=sigma, eig=eig, psi_raw=psi_raw,
        B=sigma[:, None] * V, F=H * lam, D=H * sigma ** 2 * psi,
        N=N, T=T, mp_edge=mp_edge, k=k,
    )

def sigma_dot(rm, x, sel=None):
    """Sigma @ x in factor form, optionally on a subset `sel` of the risk-model names."""
    B = rm["B"] if sel is None else rm["B"][sel]
    D = rm["D"] if sel is None else rm["D"][sel]
    return B @ (rm["F"] * (B.T @ x)) + D * x

### PCA diagnostics on one date

In [ ]:
dd = pd.Timestamp(DIAG_DATE) if DIAG_DATE else rebal_dates[-1]
dd = max(d for d in rebal_dates if d <= dd)
rm = risk_model(dd, sig_by_date[dd].index)
eig, n_show = rm["eig"], 60
share = eig / eig.sum()
K = rm["k"]
print(f"date {dd.date()}: N = {rm['N']}, T = {rm['T']}, MP edge = {rm['mp_edge']:.2f}, "
      f"k = eigenvalues above edge = {K}, variance explained by k factors: {share[:K].sum():.1%}")

fig, ax = plt.subplots()
ax.bar(np.arange(1, n_show + 1), eig[:n_show], color=C_CAP, width=0.7)
ax.axhline(rm["mp_edge"], color=C_REF, linestyle="--", linewidth=1)
ax.text(n_show, rm["mp_edge"], f"Marchenko–Pastur edge {rm['mp_edge']:.1f}  ", ha="right", va="bottom", color=C_REF)
ax.axvline(K + 0.5, color=C_REF, linestyle=":", linewidth=1)
ax.text(K + 0.7, ax.get_ylim()[1] * 0.9, f"k = {K}", color=C_REF)
ax.set_yscale("log"); ax.set_title(f"Correlation eigenvalues, {dd.date()} (log scale)")
ax.set_xlabel("Component"); ax.set_ylabel("Eigenvalue")
plt.show()

fig, ax = plt.subplots()
ax.plot(np.arange(1, n_show + 1), np.cumsum(share[:n_show]), color=C_CAP, marker="o", markersize=3)
ax.axvline(K, color=C_REF, linestyle=":", linewidth=1)
ax.text(K + 0.5, np.cumsum(share)[K - 1], f"{np.cumsum(share)[K - 1]:.0%} at k = {K}", color=C_REF, va="top")
ax.set_title("Cumulative share of variance explained"); ax.set_xlabel("Number of components")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.show()

fig, ax = plt.subplots()
ax.hist(rm["psi_raw"], bins=50, color=C_CAP, edgecolor="#fcfcfb")
ax.axvline(PSI_FLOOR, color=C_REF, linestyle="--", linewidth=1)
ax.text(PSI_FLOOR, ax.get_ylim()[1] * 0.95, f"  floor {PSI_FLOOR}", color=C_REF, va="top")
ax.set_title(r"Specific share of variance $\psi_i$ before the floor"); ax.set_xlabel(r"$\psi_i$"); ax.set_ylabel("Number of stocks")
plt.show()
print(f"stocks below the floor: {(rm['psi_raw'] < PSI_FLOOR).sum()} of {rm['N']}")

## 7. Optimizer

With active weights $a = w - b$:

$$\max_w\; \mu^\top w \;-\; \lambda\,(w-b)^\top \Sigma\,(w-b)$$

$$\text{s.t.}\quad \ell_i \le w_i \le u_i,\qquad \sum_i w_i = 1,\qquad \ell_i = \max(0,\, b_i - c),\quad u_i = b_i + c \quad (c = 2\%)$$

and $\ell_i = u_i = 0$ for names that can't be held (non-compliant names in the S&P proxy).

**Turnover penalty.** With $p$ = last month's weights drifted by this month's returns (what is actually held before trading):

$$\max_w\; \mu^\top w - \lambda (w-b)^\top\Sigma(w-b) - \kappa \sum_i \lvert w_i - p_i\rvert$$

$\kappa$ is the one-way trading cost per unit of weight, in the same monthly-return units as $\mu$: a trade is made only if its expected gain beats its cost. $\kappa = 0$ gives the problem without penalty.

A trade on an $h$-month signal pays off over about $h$ months, while $\mu$ is a one-month expected return, so the cost is spread over the horizon: $\kappa = c / h$ with $c$ = 10 bps. (Charging the full $c$ against one month of $\mu$ over-penalized trading: turnover halved but excess fell from +4.4% to +3.6%/yr; with $c/h$ excess rose to +4.65%.)

The problem is convex. We minimize $f(w) = -\mu^\top w + \lambda (w-b)^\top\Sigma(w-b)$ by **projected gradient with Nesterov acceleration (FISTA)**:

$$\nabla f(w) = -\mu + 2\lambda\,\Sigma(w-b), \qquad w^{+} = \Pi\!\left(y - \tfrac{1}{L}\nabla f(y)\right), \qquad L = 2\lambda\,\lambda_{\max}(\Sigma)$$

$\lambda_{\max}(\Sigma)$ comes from power iteration. $\Sigma x$ is always computed as $B\,(F \odot B^\top x) + D \odot x$.

**Projection** $\Pi$ onto $\{\ell \le w \le u,\ \mathbf 1^\top w = 1\}$: find the scalar $\tau$ such that

$$\sum_i \operatorname{clip}(v_i - \tau,\ \ell_i,\ u_i) = 1$$

by bisection ($\tau$ is monotone), then $w = \operatorname{clip}(v - \tau, \ell, u)$.

With the turnover penalty the step becomes a proximal step: each coordinate is first shrunk toward $p_i$ by the soft-threshold $\theta = \kappa / L$, then clipped, and $\tau$ is again found by bisection:

$$w_i(\tau) = \operatorname{clip}\!\Big(p_i + \operatorname{sign}(d_i)\max(\lvert d_i\rvert - \theta, 0),\ \ell_i,\ u_i\Big), \qquad d_i = v_i - \tau - p_i, \qquad \sum_i w_i(\tau) = 1$$

**Name cap and minimum weight** (heuristic on top of the convex solve):
1. Solve. If more than 100 names are held, keep the 100 largest (plus any name forced in by $\ell_i > 0$), set $u_i = 0$ for the rest, re-solve.
2. Names held below 0.5% get $u_i = 0$, re-solve (only if the remaining names can still sum to 1).
3. Stop when nothing changes.

In [ ]:
def project(v, lo, hi, iters=100):
    a, b_ = (v - hi).min(), (v - lo).max()
    for _ in range(iters):
        tau = 0.5 * (a + b_)
        if np.clip(v - tau, lo, hi).sum() > 1.0:
            a = tau
        else:
            b_ = tau
    return np.clip(v - 0.5 * (a + b_), lo, hi)

def lambda_max(rm, sel, iters=100):
    x = np.ones(sel.sum()) / np.sqrt(sel.sum())
    for _ in range(iters):
        y = sigma_dot(rm, x, sel)
        x = y / np.linalg.norm(y)
    return float(x @ sigma_dot(rm, x, sel))

def prox_project(v, lo, hi, p, thr, iters=100):
    """argmin_w 1/2||w - v||^2 + thr * ||w - p||_1  s.t. lo <= w <= hi, sum(w) = 1 (thr = 0 -> plain projection)."""
    if thr <= 0:
        return project(v, lo, hi, iters)
    def w_of(tau):
        d = v - tau - p
        return np.clip(p + np.sign(d) * np.maximum(np.abs(d) - thr, 0.0), lo, hi)
    a = (v - p).min() - thr - (hi - p).max() - 1.0
    b_ = (v - p).max() + thr - (lo - p).min() + 1.0
    for _ in range(iters):
        tau = 0.5 * (a + b_)
        if w_of(tau).sum() > 1.0:
            a = tau
        else:
            b_ = tau
    return w_of(0.5 * (a + b_))

def solve_qp(mu, b, lo, hi, rm, sel, lam, L, p=None, kappa=0.0, max_iter=20000, tol=1e-8):
    if p is None:
        p, kappa = np.zeros(len(b)), 0.0
    thr = kappa / L
    def f(w): return -mu @ w + lam * (w - b) @ sigma_dot(rm, w - b, sel) + kappa * np.abs(w - p).sum()
    def grad(w): return -mu + 2 * lam * sigma_dot(rm, w - b, sel)
    x = project(b.copy(), lo, hi)
    y, tk, fx = x.copy(), 1.0, f(x)
    for it in range(max_iter):
        x_new = prox_project(y - grad(y) / L, lo, hi, p, thr)
        f_new = f(x_new)
        if f_new > fx:                       # adaptive restart: fall back to a plain proximal step
            y, tk = x, 1.0
            x_new = prox_project(x - grad(x) / L, lo, hi, p, thr)
            f_new = f(x_new)
        t_new = 0.5 * (1 + np.sqrt(1 + 4 * tk ** 2))
        y = x_new + ((tk - 1) / t_new) * (x_new - x)
        done = np.abs(x_new - x).max() < tol
        x, fx, tk = x_new, min(f_new, fx), t_new
        if done:
            break
    return x, it + 1

def optimize(mu, b, rm, sel, can_hold=None, lam=LAMBDA, p=None, kappa=TURNOVER_COST):
    if can_hold is None:
        can_hold = np.ones(len(b), bool)
    lo = np.where(can_hold, np.maximum(0.0, b - ACTIVE_CAP), 0.0)
    hi = np.where(can_hold, b + ACTIVE_CAP, 0.0)
    L = 2 * lam * lambda_max(rm, sel)
    allowed = np.ones(len(b), bool)
    total_iters = 0
    for _ in range(10):
        w, n_it = solve_qp(mu, b, np.where(allowed, lo, 0), np.where(allowed, hi, 0), rm, sel, lam, L, p, kappa)
        total_iters += n_it
        held = w > 1e-9
        if held.sum() > MAX_NAMES:
            forced = lo > 0
            rest = np.where(~forced)[0]
            top = rest[np.argsort(-w[rest])][: max(MAX_NAMES - forced.sum(), 0)]
            new_allowed = forced.copy(); new_allowed[top] = True
            allowed = new_allowed
            continue
        small = held & (w < MIN_WEIGHT) & (lo == 0)
        if small.any() and hi[allowed & ~small].sum() >= 1.0 + 1e-9:
            allowed &= ~small
            continue
        break
    return w, total_iters

## 8. Run over all rebalance dates

For each date and variant we record the weights and:

- ex-ante tracking error $\mathrm{TE} = \sqrt{12\,(w-b)^\top\Sigma(w-b)}$ (annualized)
- expected active return $\mu^\top(w-b)$ (monthly)
- number of names held, PCA diagnostics

In [ ]:
weights, diag = [], []
bench_seq = {"cap": [], "ew": [], "spx": []}
prev_w = {}          # variant -> pd.Series of last target weights (for the turnover penalty)
t_start = time.time()
for n, t in enumerate(rebal_dates):
    y_hat = sig_by_date[t]
    rm = risk_model(t, y_hat.index)
    s_t = s_series.get(t, np.nan)
    beta_t = beta_series.get(t, np.nan)
    if rm is None or not np.isfinite(s_t) or not np.isfinite(beta_t):
        print(f"{t.date()}: skipped (risk model, s or beta unavailable)")
        continue
    syms = pd.Index(rm["symbols"])
    mcap = mcap_by_date.get(t, pd.Series(dtype=float))
    inv = syms.isin(list(compliant_at(t))) & syms.isin(mcap.index)
    if inv.sum() < 30:
        print(f"{t.date()}: skipped (only {inv.sum()} investable names)")
        continue
    mu_all = beta_t * s_t * y_hat.reindex(syms).fillna(0.0).to_numpy() / HORIZON_M
    cap = mcap.reindex(syms[inv]).to_numpy()
    # S&P 500 proxy: top SPX_N by market cap among all names in the risk model, compliant or not
    mcap_all = mcap_all_by_date.get(t, pd.Series(dtype=float)).reindex(syms)
    spx = syms.isin(mcap_all.dropna().nlargest(SPX_N).index)
    sel_spx = inv | spx
    b_spx = np.where(spx[sel_spx], mcap_all.to_numpy()[sel_spx], 0.0)
    cases = {
        "cap": (inv, cap / cap.sum(), None),
        "ew":  (inv, np.full(inv.sum(), 1.0 / inv.sum()), None),
        "spx": (sel_spx, b_spx / b_spx.sum(), inv[sel_spx]),
    }
    share = rm["eig"] / rm["eig"].sum()
    month_ret = np.exp(logret_m.loc[t.to_period("M")]) - 1 if t.to_period("M") in logret_m.index else None
    for variant, (sel, b, can_hold) in cases.items():
        sel_syms = syms[sel]
        mu = mu_all[sel]
        # p = last month's weights drifted by this month's returns (what is held before trading)
        p_full = prev_w.get(variant)
        if p_full is not None and month_ret is not None:
            p_full = p_full * (1 + month_ret.reindex(p_full.index).fillna(0.0))
            p_full = p_full / p_full.sum()
        p = p_full.reindex(sel_syms).fillna(0.0).to_numpy() if p_full is not None else None
        w, iters = optimize(mu, b, rm, sel, can_hold, p=p)
        turnover = (0.5 * (np.abs(w - p).sum() + p_full.drop(sel_syms, errors="ignore").sum())
                    if p_full is not None else np.nan)
        prev_w[variant] = pd.Series(w, index=sel_syms)[w > 1e-9]
        a = w - b
        diag.append(dict(
            date=t, variant=variant, n_investable=int(inv.sum()), n_bench=int((b > 0).sum()), n_names=int((w > 1e-9).sum()),
            te_ann=np.sqrt(12 * a @ sigma_dot(rm, a, sel)), exp_active_ret_m=mu @ a,
            max_w=w.max(), turnover=turnover, s=s_t, beta=beta_t, n_pca=rm["N"], k=rm["k"],
            var_expl_k=share[:rm["k"]].sum(), iters=iters,
        ))
        bench_seq[variant].append((t, {k: v for k, v in zip(sel_syms, b) if v > 0}))
        held = w > 1e-9
        weights.append(pd.DataFrame({
            "date": t.strftime("%Y-%m"), "symbol": sel_syms[held], "weight": w[held],
            "benchmark_weight": b[held], "inference_date": t.strftime("%Y-%m-%d"), "variant": variant,
        }))
    if n % 12 == 0:
        print(f"{t.date()}  done {n + 1}/{len(rebal_dates)}  elapsed {time.time() - t_start:.0f}s")

weights = pd.concat(weights, ignore_index=True)
diag = pd.DataFrame(diag)
print(f"finished in {time.time() - t_start:.0f}s")
diag.groupby("variant")[["n_investable", "n_bench", "n_names", "turnover", "te_ann", "exp_active_ret_m", "max_w", "iters"]].median()

### Diagnostics over time

In [ ]:
d = diag.pivot(index="date", columns="variant")

fig, ax = plt.subplots()
ax.plot(d.index, d["te_ann"]["cap"], color=C_CAP, label="Cap-weighted benchmark")
ax.plot(d.index, d["te_ann"]["ew"], color=C_EW, label="Equal-weighted benchmark")
ax.plot(d.index, d["te_ann"]["spx"], color=C_SPX, label="S&P 500 proxy benchmark")
ax.set_title(f"Ex-ante tracking error (annualized), λ = {LAMBDA}")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}")); ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(d.index, d["n_names"]["cap"], color=C_CAP, label="Cap-weighted benchmark")
ax.plot(d.index, d["n_names"]["ew"], color=C_EW, label="Equal-weighted benchmark")
ax.plot(d.index, d["n_names"]["spx"], color=C_SPX, label="S&P 500 proxy benchmark")
ax.set_title("Number of names held"); ax.set_ylabel("Names"); ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(d.index, d["k"]["cap"], color=C_CAP)
ax.set_title("Number of factors k (eigenvalues above the Marchenko–Pastur edge)"); ax.set_ylabel("k")
plt.show()

fig, ax = plt.subplots()
ax.plot(d.index, d["var_expl_k"]["cap"], color=C_CAP)
ax.set_title("Share of variance explained by the k factors")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.show()

## 9. Export

One CSV per variant in the same format as the live weight files (`date, symbol, weight, inference_date`), plus `benchmark_weight` for reference, and the diagnostics table.

Also `benchmark_cap.pkl` / `benchmark_ew.pkl`: the full benchmark weights $b$ in the same format, to simulate the benchmark itself (strategy − own benchmark = what the signal adds).

Also one pickle per variant holding the backtest's `weight_sequence` format: a list of `(period_date, {symbol: weight})`, ready for `simulate_rebalanced_portfolio`.

In [ ]:
for variant, g in weights.groupby("variant"):
    path = OUT_DIR / f"weights_{variant}.csv"
    g.drop(columns="variant").to_csv(path, index=False)
    print(f"wrote {path}  ({g['date'].nunique()} dates, {len(g)} rows)")
    seq = [(pd.Timestamp(d), dict(zip(x["symbol"], x["weight"])))
           for d, x in g.groupby("inference_date")]
    pd.to_pickle(seq, OUT_DIR / f"weight_sequence_{variant}.pkl")
diag.to_csv(OUT_DIR / "diagnostics.csv", index=False)
for variant, seq in bench_seq.items():
    pd.to_pickle(seq, OUT_DIR / f"benchmark_{variant}.pkl")
    print(f"wrote {OUT_DIR / f'benchmark_{variant}.pkl'}  ({len(seq)} dates)")

last = weights["inference_date"].max()
(weights[weights["inference_date"] == last]
   .assign(active=lambda x: x["weight"] - x["benchmark_weight"])
   .sort_values(["variant", "weight"], ascending=[True, False])
   .groupby("variant").head(15))